# 01 - Ingestao Pix

Este notebook coleta dados publicos reais do Pix, valida a origem e salva a camada Bronze em formato Parquet.

## Conceitos

Ingestao e a etapa responsavel por trazer dados de uma fonte externa para o ambiente analitico. A camada Bronze armazena os dados brutos, com o minimo de alteracao possivel. O formato Parquet e utilizado por ser colunar, eficiente e adequado para processamento distribuido com Spark.

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().resolve().parent if Path.cwd().resolve().name in {"notebooks", "i_notebooks"} else Path.cwd().resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))


In [ ]:
from src.config import PIX_RAW_DIR, PIX_RAW_SAMPLE_CSV, create_project_directories
from src.data_quality import ensure_not_empty
from src.data_source import load_public_pix_data
from src.spark_session import get_spark_session

create_project_directories(verbose=True)
spark = get_spark_session("01-ingestion-pix")

In [ ]:
source_result = load_public_pix_data()
print(f"Origem utilizada: {source_result.source_type}")
print(f"Referencia da fonte: {source_result.source_reference}")
print(f"Quantidade de registros carregados: {len(source_result.data)}")

pix_raw_df = spark.createDataFrame(source_result.data)
ensure_not_empty(pix_raw_df, "Bronze Pix raw")

In [ ]:
pix_raw_df.printSchema()
pix_raw_df.show(10, truncate=False)
print(f"Registros na camada Bronze a gravar: {pix_raw_df.count()}")

In [ ]:
pix_raw_df.write.mode("overwrite").parquet(str(PIX_RAW_DIR))
pix_raw_df.coalesce(1).write.mode("overwrite").option("header", True).csv(str(PIX_RAW_SAMPLE_CSV))
print(f"Camada Bronze gravada em: {PIX_RAW_DIR.relative_to(PROJECT_DIR)}")
print(f"Copia CSV de conferencia gravada em: {PIX_RAW_SAMPLE_CSV.relative_to(PROJECT_DIR)}")

In [ ]:
spark.stop()